In [ ]:
import os, re, pandas as pd
from scipy.stats import loguniform
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.feature_extraction.text import CountVectorizer

paths = [
    "./datasets/phishing_transformed.csv","../datasets/phishing_transformed.csv",
    "./datasets/phishing.csv","../datasets/phishing.csv","../../datasets/phishing.csv",
    "phishing_transformed.csv","phishing.csv"
]
csv = next((p for p in paths if os.path.exists(p)), None)
if not csv: raise FileNotFoundError("Coloque phishing_transformed.csv ou phishing.csv em ./datasets/")

try: df = pd.read_csv(csv, encoding="utf-8")
except UnicodeDecodeError: df = pd.read_csv(csv, encoding="latin1")

if "Email Type_Phishing Email" in df.columns:
    y = df["Email Type_Phishing Email"].astype(int)
elif "Email Type" in df.columns:
    y = df["Email Type"].astype(str).str.lower().str.contains("phishing").astype(int)
else:
    raise ValueError("Alvo de phishing não encontrado.")

def clean(s):
    s = str(s).lower()
    s = re.sub(r'[^a-zA-Z0-9áéíóúãõâêôçÁÉÍÓÚÃÕÂÊÔÇ\s]',' ',s)
    s = re.sub(r'\b\d+\b',' ',s)
    return re.sub(r'\s+',' ',s).strip()

if "Email Text" in df.columns:
    df["Email Text"] = df["Email Text"].apply(clean)
    num = df.drop(columns=[c for c in ["Email Text","Email Type","Email Type_Phishing Email"] if c in df.columns]) \
            .select_dtypes(include=["number"]).columns
    prep = ColumnTransformer([
        ("text", CountVectorizer(max_features=4000, min_df=5, max_df=0.8), "Email Text"),
        ("num",  StandardScaler(with_mean=False), num)
    ], remainder="drop")
    X = df
else:
    feat = [c for c in df.columns if c not in ["Email Type","Email Type_Phishing Email"]]
    X = df[feat]
    num = X.select_dtypes(include=["number"]).columns
    prep = ColumnTransformer([("num", StandardScaler(with_mean=False), num)], remainder="drop")

pipe = Pipeline([("prep", prep), ("clf", SVC(probability=True, class_weight='balanced', random_state=42))])

cv = StratifiedKFold(5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, scoring="accuracy", cv=cv, n_jobs=-1)
print(f"[KFold-5] acc: {scores.mean():.4f} ± {scores.std():.4f}")

param_dist = {
    "clf__kernel": ["rbf", "linear"],
    "clf__C": loguniform(1e-2, 1e3),
    "clf__gamma": ["scale", "auto"]
}
rs = RandomizedSearchCV(pipe, param_dist, n_iter=20, scoring="accuracy",
                        cv=cv, n_jobs=-1, random_state=42, verbose=1)
rs.fit(X, y)
print(f"[RandomSearch] best_acc: {rs.best_score_:.4f}\nparams: {rs.best_params_}")
